# Post-processing and aggregation of detections

In [ ]:
from datetime import datetime
import json
import os
import requests

import geopandas as gpd
import shapely.geometry as sg

from zwerfafval_detectie.utils_eval import read_annotations_folder

RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

In [ ]:
date = "250514"

predictions_folder = f"../datasets/experiments/zwerfafval/predict_v1_extra_2/inwinning_{date}_26m_1920"
output_folder = "../datasets/experiments/zwerfafval/heatmap_new"

city_geojson = "../datasets/experiments/zwerfafval/gebieden_v1_buurten_openbaar.geojson"

if date == "250514":
    metadata_file = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/frames_gdf.gpkg"
elif date == "260421":
    metadata_file = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/frames_1fps_gdf.gpkg"
else:
    metadata_json_folder = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/json_tasks"

categories = {
    0: "Zwerfafval (grof)",
    1: "Zwerfafval (fijn)"
}

confidence = 0.3

os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Load model predictions

predictions_gdf = read_annotations_folder(folder_path=predictions_folder, categories=categories)
predictions_gdf["file_name"] = predictions_gdf["file_name"].apply(lambda f: os.path.splitext(f)[0])

_predictions_sorted = (
    predictions_gdf[predictions_gdf["confidence"] >= confidence]
    .set_index("file_name")
    .sort_index()
)


# Count predictions per image

counts_df = (
    _predictions_sorted[["category"]]
    .replace(categories)
    .groupby(["file_name", "category"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
if date in {"250514", "260421"}:
    # Load metadata file (for 260421 or 250514)
    metadata_gdf = gpd.read_file(metadata_file)
    metadata_gdf["file_name"] = metadata_gdf["file_name"].apply(lambda f: os.path.splitext(f)[0])
    metadata_gdf.rename(columns={"frame_timestamp": "timestamp_utc"}, inplace=True)
    metadata_gdf = metadata_gdf[["file_name", "timestamp_utc", "geometry"]].set_index("file_name")
else:
    # Load metadata from JSON files

    data = {
        "file_name": [],
        "timestamp_utc": [],
        "geometry": [],
    }

    metadata_files = sorted([
        file for file in os.listdir(metadata_json_folder) 
        if os.path.splitext(file)[1] == ".json"
    ])

    for file in metadata_files:
        with open(os.path.join(metadata_json_folder, file), 'r') as fh:
            json_content = json.load(fh)
            data["file_name"].append(
                os.path.splitext(json_content["image_file_name"])[0]
            )
            data["timestamp_utc"].append(
                datetime.fromisoformat(json_content["image_file_timestamp"])
            )
            data["geometry"].append(
                sg.Point((
                    json_content["gps_data"]["longitude"],
                    json_content["gps_data"]["latitude"]
                ))
            )

    metadata_gdf = gpd.GeoDataFrame(
        data=data,
        crs=LAT_LON_CRS
    ).set_index("file_name")

In [ ]:
# Merge object counts and metadata

counts_merged = (
    gpd.GeoDataFrame(counts_df.join(metadata_gdf, how="outer"))
    .fillna(value={
        "Zwerfafval (fijn)": 0,
        "Zwerfafval (grof)": 0,
    })
    .to_crs(RD_CRS)
)

In [ ]:
# Discard detections outside of Amsterdam (GPS glitches)

amsterdam_shape = gpd.read_file(city_geojson).to_crs(RD_CRS).union_all()

counts_merged = counts_merged[counts_merged.within(amsterdam_shape)]

In [ ]:
# Add street and park info

street_url = "https://maps.amsterdam.nl/open_geodata/geojson_lnglat.php?KAARTLAAG=STRAATNAMEN&THEMA=straatnamen"
response = requests.get(street_url)
streets_gdf = gpd.GeoDataFrame.from_features(response.json()["features"], crs=LAT_LON_CRS)

parks_url = "https://maps.amsterdam.nl/open_geodata/geojson_lnglat.php?KAARTLAAG=PARKPLANTSOENGROEN&THEMA=stadsparken"
response = requests.get(parks_url)
parks_gdf = gpd.GeoDataFrame.from_features(response.json()["features"], crs=LAT_LON_CRS)


# Merge street
counts_merged = (
    counts_merged
    .sjoin_nearest(
        right=streets_gdf[["STT_NAAM", "geometry"]].to_crs(RD_CRS), 
        how="left", 
        distance_col="dist_to_street"
    )
    .drop(columns="index_right")
    .rename(columns={"STT_NAAM": "straat_naam"})
)

# Merge park
counts_merged = (
    counts_merged
    .sjoin_nearest(
        right=parks_gdf[["Naam", "geometry"]].to_crs(RD_CRS), 
        how="left", 
        distance_col="dist_to_park"
    )
    .drop(columns="index_right")
    .rename(columns={"Naam": "park_naam"})
)

In [ ]:
# Select image if distance to previous selected image is larger than a threshold

min_distance = 5.0

points = counts_merged["geometry"].to_crs(RD_CRS)

previous = points.iloc[0]

selection = [False]*len(points)
selection[0] = True

for i, point in enumerate(points.iloc[1:]):
    if previous.distance(point) >= min_distance:
        previous = point
        selection[i+1] = True

counts_merged["selected"] = False
counts_merged.loc[:, "selected"] = selection

In [ ]:
# Compute average counts for each selected image by averaging all upcoming
# images until the next selected 

# The assumption is that the camera looks forward
# so the upcoming images are a good representation for the current situation

counts_merged.reset_index(inplace=True)
idx_selected = counts_merged.index[counts_merged["selected"]].tolist()

for i in range(0, len(idx_selected) - 1):
    start_idx = idx_selected[i]
    end_idx = idx_selected[i+1] - 1
    section_mean_grof = counts_merged.loc[start_idx:end_idx, "Zwerfafval (grof)"].mean()
    section_mean_fijn = counts_merged.loc[start_idx:end_idx, "Zwerfafval (fijn)"].mean()
    counts_merged.loc[start_idx, "section_mean_grof"] = section_mean_grof
    counts_merged.loc[start_idx, "section_mean_fijn"] = section_mean_fijn

counts_merged.set_index("file_name", inplace=True)

# If the last row was "selected", it won't have a mean so we unselect it
counts_merged.loc[counts_merged.index[-1], "selected"] = False

In [ ]:
# Compute rolling average over section counts
# A window of 5 means averaging over 25m stretches (since each section is 5m)

counts_merged.loc[counts_merged["selected"], "rolling_avg_grof"] = (
    counts_merged.loc[counts_merged["selected"], "section_mean_grof"]
    .rolling(window=5, min_periods=1, center=True).mean()
)

counts_merged.loc[counts_merged["selected"], "rolling_avg_fijn"] = (
    counts_merged.loc[counts_merged["selected"], "section_mean_fijn"]
    .rolling(window=5, min_periods=1, center=True).mean()
)

In [ ]:
counts_merged.to_file(
    filename=os.path.join(output_folder, f"counts_{date}.gpkg"),
    driver="GPKG"
)

In [ ]:
# Plot results on a map

from xyzservices import TileProvider

rolling = False

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

if rolling:
    plot_column = "rolling_avg_grof"
else:
    plot_column = "section_mean_grof"

map = (
    counts_merged[counts_merged["selected"]].dropna()
    .explore(
        column=plot_column,
        cmap="YlOrRd",
        style_kwds={
            "style_function": lambda x: {"radius": 2*x["properties"][plot_column]},
            "fillOpacity": 0.75,
            "weight": 2
        },
        legend=True,
        tiles=ams_tile_provider
    )
)

if rolling:
    map.save(os.path.join(output_folder, f"heatmap_inwinning_{date}_sampled_roll.html"))
else:
    map.save(os.path.join(output_folder, f"heatmap_inwinning_{date}_sampled.html"))

## Top 10

In [ ]:
counts_merged = gpd.read_file(
    filename=os.path.join(output_folder, f"counts_{date}.gpkg")
)

In [ ]:
top10_dirty = counts_merged.sort_values("Zwerfafval (grof)", ascending=False).iloc[0:10, :]

In [ ]:
top10_dirty

In [ ]:
top10_clean = counts_merged[counts_merged["Zwerfafval (grof)"]==0].sample(10)

In [ ]:
top10_clean

In [ ]:
import shutil

if date=="260824":
    images_folder = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/recording_2026_08_24T13-17-29+0200/images/"
elif date=="260713":
    images_folder = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/recording_2026_07_13T09-50-35+0200/images/"
elif date=="250514":
    images_folder = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/images/"

_output_folder = os.path.join(output_folder, f"top10_{date}")
_output_folder_dirty = os.path.join(_output_folder, "vies")
_output_folder_clean = os.path.join(_output_folder, "schoon")

os.makedirs(_output_folder_dirty, exist_ok=True)
os.makedirs(_output_folder_clean, exist_ok=True)

top10_dirty.to_file(os.path.join(_output_folder, "top10_dirty.gpkg"), driver="GPKG")
top10_clean.to_file(os.path.join(_output_folder, "top10_clean.gpkg"), driver="GPKG")

for name in top10_dirty["file_name"]:
    img_src = os.path.join(images_folder, f"{name}.jpg")
    label_src = os.path.join(predictions_folder, f"{name}.txt")
    shutil.copy2(img_src, _output_folder_dirty)
    if os.path.exists(label_src):
        shutil.copy2(label_src, _output_folder_dirty)

for name in top10_clean["file_name"]:
    img_src = os.path.join(images_folder, f"{name}.jpg")
    label_src = os.path.join(predictions_folder, f"{name}.txt")
    shutil.copy2(img_src, _output_folder_clean)
    if os.path.exists(label_src):
        shutil.copy2(label_src, _output_folder_clean)

## Sample streets

In [ ]:
counts_merged = gpd.read_file(
    filename=os.path.join(output_folder, f"counts_{date}.gpkg")
)

In [ ]:
counts_merged = counts_merged[counts_merged["selected"]]
counts_merged = counts_merged[counts_merged["dist_to_street"]<=10.0]

In [ ]:
straten = [
    "Polanenstraat",
    "Ferdinand Huyckstraat",
    "Leeuwendalersweg",
    "Wormerveerstraat",
]

In [ ]:
straat_counts = counts_merged[counts_merged["straat_naam"].isin(straten)]

In [ ]:
straat_counts.groupby("straat_naam")["section_mean_grof"].mean()

In [ ]:
# Copy images per street into separate folders

import re
import shutil
import unicodedata

def slugify(value, allow_unicode=False):
    """
    Taken from https://github.com/django/django/blob/master/django/utils/text.py
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize('NFKC', value)
    else:
        value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub(r'[^\w\s-]', '', value.lower())
    return re.sub(r'[-\s]+', '-', value).strip('-_')


images_folder = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/recording_2026_08_24T13-17-29+0200/images/"
straten_output_folder = os.path.join(output_folder, f"stuktelling_{date}")

for straat in straten:
    _straat_output = os.path.join(straten_output_folder, slugify(straat))
    os.makedirs(_straat_output, exist_ok=True)
    _straat_df = straat_counts[straat_counts["straat_naam"]==straat]
    for name in _straat_df["file_name"]:
        img_file = os.path.join(images_folder, f"{name}.jpg")
        shutil.copy2(img_file, _straat_output)

## Compare counts to manual counts per image (stuktellingen)

In [ ]:
fotos = [
    "20260824_115259_703753_024234",
    "20260824_115301_364594_024248",
    "20260824_115302_976784_024262",
    "20260824_115304_555718_024276",
    "20260824_115306_173467_024290",
    "20260824_115307_870733_024304",
    "20260824_115309_486215_024318",
    "20260824_115311_082100_024332",
    "20260824_115312_705076_024346",
    "20260824_115315_442120_024374",
    "20260824_115317_821997_024409",
    "20260824_115319_731343_024437",
    "20260824_115321_672930_024465",
    "20260824_115323_995767_024500",
    "20260824_115326_339034_024535",
    "20260824_115347_301271_024843",
    "20260824_115348_719087_024864",
    "20260824_115350_145751_024885",
    "20260824_115351_615779_024906",
    "20260824_115353_060778_024927",
    "20260824_115354_482122_024948",
    "20260824_115355_923797_024969",
    "20260824_115357_341460_024990",
    "20260824_115358_840759_025011",
    "20260824_115400_278789_025032",
    "20260824_115401_726813_025053",
    "20260824_115403_130219_025074",
    "20260824_115404_552395_025095",
    "20260824_115405_968143_025116",
    "20260824_115407_414604_025137",
    "20260824_142247_559772_128905",
    "20260824_142249_100975_128926",
    "20260824_142250_558442_128947",
    "20260824_142340_083494_129661",
    "20260824_142341_602293_129675",
    "20260824_142343_259475_129689",
    "20260824_142344_928809_129703",
    "20260824_142349_118489_129738",
    "20260824_142350_752433_129752",
    "20260824_142350_752433_129752",
    "20260824_142354_157407_129780",
    "20260824_142355_832017_129794",
    "20260824_142357_625852_129808",
    "20260824_142359_205124_129822",
    "20260824_142400_869899_129836",
    "20260824_141833_296479_126504",
    "20260824_141835_138594_126532",
    "20260824_141836_588614_126553",
    "20260824_141838_030753_126574",
    "20260824_141839_446171_126595",
    "20260824_141840_864227_126616",
    "20260824_141842_269468_126637",
    "20260824_141843_687259_126658",
    "20260824_141845_113279_126679",
    "20260824_141846_550497_126700",
    "20260824_141848_022488_126721",
    "20260824_141849_477184_126742",
    "20260824_141850_952973_126763",
    "20260824_141852_412260_126784",
    "20260824_141853_856748_126805",
]

In [ ]:
straat_counts.loc[fotos, ["Zwerfafval (grof)", "Zwerfafval (fijn)", "section_mean_grof", "rolling_avg_grof"]]

In [ ]:
# Copy photos and labels into folders

import shutil

images_folder = f"../datasets/experiments/zwerfafval/data_inwinning_{date}/recording_2026_08_24T13-17-29+0200/images/"
straten_output_folder = os.path.join(output_folder, f"stuktelling_{date}")
images_output_folder = os.path.join(straten_output_folder, "images")
labels_output_folder = os.path.join(straten_output_folder, "labels")

os.makedirs(images_output_folder, exist_ok=True)
os.makedirs(labels_output_folder, exist_ok=True)

for foto in fotos:
    img_file = os.path.join(images_folder, f"{foto}.jpg")
    label_file = os.path.join(predictions_folder, f"{foto}.txt")
    shutil.copy2(img_file, images_output_folder)
    if os.path.exists(label_file):
        shutil.copy2(label_file, labels_output_folder)